In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# 한글 폰트 설정 (Mac용: AppleGothic, Windows용: Malgun Gothic, Linux용: NanumGothic 등)
mpl.rcParams['font.family'] = 'AppleGothic'       # 또는 'Malgun Gothic', 'NanumGothic'
mpl.rcParams['axes.unicode_minus'] = False        # 마이너스 기호 깨짐 방지

In [ ]:
import pandas as pd
# phase2 
df1 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/비트코인.csv')
df2 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/이더리움.csv')
df3 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/채권.csv')
# etf = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/국내ETF.csv', encoding = enc)
# kospi_tpo100 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv')

In [ ]:
import chardet

# 파일 앞부분만 읽어서 인코딩 추측
raw = open('/Users/qwergs/Desktop/필드/phase2/국내ETF.csv', 'rb').read(10000)
enc = chardet.detect(raw)['encoding']
print("추측된 인코딩:", enc)

# 추측된 인코딩으로 다시 읽기
df4 = pd.read_csv(
    '/Users/qwergs/Desktop/필드/phase2/국내ETF.csv',
    encoding=enc
)

In [ ]:
import chardet

# 파일 앞부분 바이트를 읽어서 인코딩 추측
raw = open('/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv','rb').read(5000)
enc = chardet.detect(raw)['encoding']
print("추측 인코딩:", enc)

# 추측된 인코딩으로 다시 읽기
df5 = pd.read_csv(
    '/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv',
    encoding=enc
)

In [ ]:
btc = df1.copy()
edh = df2.copy()
bond = df3.copy()
etf = df4.copy()
kospi = df5.copy()

In [ ]:
btc.columns

In [ ]:
edh.columns

In [ ]:
bond.columns

In [ ]:
etf.columns

In [ ]:
etf['ETF_Name'].value_counts()

In [ ]:
etf["ETF_Name"].unique()

In [ ]:
# ETF 데이터를 Date 기준으로 ETF_Name을 컬럼으로 피벗
etf_pivot = etf.pivot(index='Date', columns='ETF_Name', values='Close')

In [ ]:
etf_pivot.columns

In [ ]:
# 중기 
col_mid = ['KODEX 3대농산물선물', 'Kodex 모멘텀PLUS', 'KODEX 종합채권(AA-이상)액티브', 'KOSEF 미국달러선물', 'TIGER CD금리투자KIS(합성) ETF']

In [ ]:
# 장기 
col_long = ['KODEX 200선물인버스2X', 'ACE KRX금현물', 'HANARO 농업융복합산업', 'KB KBSTAR 글로벌 수소경제 Index',
            'KBSTAR 글로벌 4차 산업혁명 IT', 'KIM ACE NASDAQ100', 'KODEX 2차전지산업', 'KODEX IT',
            'Kodex 미국30년국채울트라선물(H)', 'KODEX 미국S&P500선물', 'KODEX 미국러셀2000', 
            'KODEX 반도체', '신한 SOL 200 Total Return', 'KODEX 코스닥150'] 

In [ ]:
# 단기 
col_short = [
    'ACE 인버스', 'KODEX 200선물인버스2X', 'ACE 인버스', 'KODEX 코스닥150 레버리지',
    'KODEX 코스닥150선물인버스', 'TIGER 미국달러단기채권액티브', 'TIGER 인버스',
    'KOSEF 미국달러선물', 'TIGER CD금리투자KIS(합성) ETF'
] 

In [ ]:
# etf_mid = etf[col_mid].copy()
# etf_long = etf[col_long].copy()
# etf_short = etf[col_short].copy()
etf_mid = etf[etf["ETF_Name"].isin(col_mid)].copy()
etf_long = etf[etf["ETF_Name"].isin(col_long)].copy()
etf_short = etf[etf["ETF_Name"].isin(col_short)].copy()

In [ ]:
# ETF 수익률 (ETF_Name별로 피벗 후 수익률 계산)
etf_short_ret = etf_short[etf_short["ETF_Name"].isin(col_short)] \
    .pivot(index='Date', columns='ETF_Name', values='Close') \
    .pct_change().dropna()

etf_mid_ret = etf_mid[etf_mid["ETF_Name"].isin(col_mid)] \
    .pivot(index='Date', columns='ETF_Name', values='Close') \
    .pct_change().dropna()

etf_long_ret = etf_long[etf_long["ETF_Name"].isin(col_long)] \
    .pivot(index='Date', columns='ETF_Name', values='Close') \
    .pct_change().dropna()

In [ ]:
def compute_etf_returns(etf_df, etf_list):
    """ETF 수익률 계산 (ETF_Name 기준 필터 + 피벗 + 수익률 계산)"""
    filtered = etf_df[etf_df["ETF_Name"].isin(etf_list)].copy()
    filtered['Date'] = pd.to_datetime(filtered['Date'])
    ret = filtered.pivot(index='Date', columns='ETF_Name', values='Close').pct_change().dropna()
    return ret

In [ ]:
etf_short_ret = compute_etf_returns(etf_short, col_short)
etf_mid_ret   = compute_etf_returns(etf_mid, col_mid)
etf_long_ret  = compute_etf_returns(etf_long, col_long)

In [ ]:
kospi.columns

In [ ]:
kospi['Name'].unique()

# 포트폴리오 최적화 모델 비교 표 (1~5)

| 번호 | 모델명                          | 핵심 개념                                                                 | 장점                                                                 | 단점                                                                 |
|------|--------------------------------|--------------------------------------------------------------------------|----------------------------------------------------------------------|----------------------------------------------------------------------|
| 1    | Mean-Variance Optimization     | 평균 수익률과 분산(위험)을 수학적으로 최적화하여 효율적 포트폴리오 구성         | - 수익과 리스크를 함께 고려<br>- 이론적 기반이 강력                 | - 수익률 예측 민감<br>- 공분산 추정 민감<br>- 계산 복잡               |
| 2    | Black-Litterman                | 시장 기대수익 + 투자자의 주관적 뷰를 결합하여 수익률 추정                         | - 시장 정보와 투자자 뷰 통합<br>- 자산 쏠림 방지                     | - 수학적 복잡성<br>- 주관적 뷰의 수치화 어려움                        |
| 3    | Risk Parity                    | 자산의 위험 기여도를 동일하게 조정하여 전체 위험 균형                               | - 수익 예측 불필요<br>- 리스크 분산 우수                             | - 채권 중심 비중 확대 가능<br>- 성장성 희생 우려                      |
| 4    | Minimum Variance Portfolio     | 전체 변동성을 최소화하는 자산 배분 전략                                            | - 하락장에 유리<br>- 예측 불확실성 회피 가능                         | - 수익 극대화 목적 아님<br>- 성과 향상 한계 있음                      |
| 5    | CVaR Optimization              | 최악의 손실 발생 시 평균 손실(CVaR)을 최소화                                       | - 보수적이고 현실적인 리스크 관리<br>- 위기 상황에 강함             | - 수학적으로 복잡<br>- 수익보다 리스크 회피에 치중될 수 있음         |

# `Markowitz, Mean Variance Portfolio(MVP)`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize

# [1] 수익률 표준화 함수 (변동성 기준 스케일링)
def scale_returns(df):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df)
    return pd.DataFrame(scaled, index=df.index, columns=df.columns)

# [2] 평균-분산 최적화 (Markowitz MVP)
def minimum_variance_portfolio(returns, max_assets=None):
    """
    Markowitz 평균-분산 최적화 모델 (MVO)
    
    목표:
        수익률은 높이고, 포트폴리오 분산(위험)은 낮추는 최적 투자 조합 도출

    수학적 개념:
        기대 수익률 E(Rp) = ∑ wi * E(Ri)
        포트폴리오 분산 σp² = wᵗ * Σ * w

    입력:
        returns (DataFrame): 수익률 데이터 (행=시점, 열=자산)
        max_assets (int): 선택할 최대 자산 수 (선택형)

    출력:
        weight_series (Series): 자산별 최적 비중
    """
    cov = returns.cov().values         # Σ: 자산 간 공분산 행렬
    n_assets = cov.shape[0]
    columns = returns.columns

    # 목적함수: 포트폴리오 분산 최소화 → σp² = wᵗΣw
    def portfolio_variance(w):
        return w.T @ cov @ w

    # 제약조건: ∑ wi = 1 (전체 투자금 합 = 100%)
    cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

    # 경계조건: 0 ≤ wi ≤ 1 (short selling 금지)
    bounds = tuple((0, 1) for _ in range(n_assets))
    x0 = np.ones(n_assets) / n_assets  # 초기 guess: 균등 분배

    # 최적화 수행
    result = minimize(portfolio_variance, x0, method='SLSQP', bounds=bounds, constraints=cons)
    weights = result.x

    # 자산별 가중치 반환
    weight_series = pd.Series(weights, index=columns)

    # 상위 N개 자산만 선택 후 재정규화 (선택적)
    if max_assets:
        weight_series = weight_series.sort_values(ascending=False).head(max_assets)
        weight_series = weight_series / weight_series.sum()

    return weight_series.sort_values(ascending=False)

In [ ]:
import pandas as pd

# 날짜 처리
btc['Date'] = pd.to_datetime(btc['Date'])
edh['Date'] = pd.to_datetime(edh['Date'])
bond['Date'] = pd.to_datetime(bond['Date'])
# etf['Date'] = pd.to_datetime(etf['Date'])
kospi['Date'] = pd.to_datetime(kospi['Date'])
etf_short['Date'] = pd.to_datetime(etf_short['Date'])
etf_mid['Date'] = pd.to_datetime(etf_mid['Date'])
etf_long['Date'] = pd.to_datetime(etf_long['Date'])

# BTC, ETH 수익률 계산
btc_ret = btc.set_index('Date')['Close'].pct_change().dropna().rename('BTC')
edh_ret = edh.set_index('Date')['Close'].pct_change().dropna().rename('ETH')

# ETF 수익률 (ETF_Name별로 피벗 후 수익률 계산)
# etf_ret = etf.pivot(index='Date', columns='ETF_Name', values='Close').pct_change().dropna()
etf_short_ret = etf_short.pivot(index='Date', columns = 'ETF_Name', values='Close').pct_change().dropna()
etf_mid_ret = etf_mid.pivot(index='Date', columns = 'ETF_Name',values='Close').pct_change().dropna()
etf_long_ret = etf_long.pivot(index='Date', columns = 'ETF_Name',values='Close').pct_change().dropna()

# KOSPI 수익률 (종목 이름 기준으로 피벗)
kospi_ret = kospi.pivot(index='Date', columns='Name', values='Close').pct_change().dropna()

# 채권 수익률 (Close가 아닌 금리라면 변동률, 가격이라면 pct_change)
bond_ret = bond.set_index('Date').pct_change().dropna()
bond_ret = bond_ret.rename(columns=lambda x: f'BOND_{x}' if x != 'Date' else x)

# 공통 날짜만 정렬
common_dates_3yr = btc_ret.index \
    .intersection(edh_ret.index) \
    .intersection(etf_short_ret.index) \
    .intersection(etf_mid_ret.index) \
    .intersection(etf_long_ret.index) \
    .intersection(kospi_ret.index)

common_dates_bond = bond_ret.index

# 전체 통합 수익률 (BTC, ETH, KOSPI만 포함)
# returns_3yr = pd.concat([
#     btc_ret.loc[common_dates_3yr],
#     edh_ret.loc[common_dates_3yr],
#     kospi_ret.loc[common_dates_3yr]
# ], axis=1).dropna(axis=1, how='any')

# ETF 단기 수익률만 통합
returns_3yr_short = pd.concat([
    etf_short_ret.loc[common_dates_3yr],
    btc_ret.loc[common_dates_3yr],
    edh_ret.loc[common_dates_3yr],
    kospi_ret.loc[common_dates_3yr]
], axis=1).dropna(axis=1, how='any')

# ETF 중기 수익률만 통합
returns_3yr_mid = pd.concat([
    etf_mid_ret.loc[common_dates_3yr],
    btc_ret.loc[common_dates_3yr],
    edh_ret.loc[common_dates_3yr],
    kospi_ret.loc[common_dates_3yr]
], axis=1).dropna(axis=1, how='any')

# ETF 장기 수익률만 통합
returns_3yr_long = pd.concat([
    etf_long_ret.loc[common_dates_3yr],
    btc_ret.loc[common_dates_3yr],
    edh_ret.loc[common_dates_3yr],
    kospi_ret.loc[common_dates_3yr]
], axis=1).dropna(axis=1, how='any')

# 채권 수익률
returns_bond = bond_ret.loc[common_dates_bond].dropna(axis=1, how='any')

In [ ]:
# [3] 설정: 총 자산 수와 비중 기반 분할
total_assets = 10
ratio_3yr = 0.4
ratio_bond = 0.6

max_assets_3yr = int(round(total_assets * ratio_3yr))
max_assets_bond = total_assets - max_assets_3yr

# [4] 수익률 표준화
returns_3yr_scaled = scale_returns(returns_3yr_short)
returns_bond_scaled = scale_returns(returns_bond)

# [5] MVP 최적화
weights_3yr = minimum_variance_portfolio(returns_3yr_scaled, max_assets=max_assets_3yr)
weights_bond = minimum_variance_portfolio(returns_bond_scaled, max_assets=max_assets_bond)

# [6] 비중 합산 및 정규화
combined_weights = pd.concat([
    weights_3yr * ratio_3yr,
    weights_bond * ratio_bond
])
combined_weights = combined_weights / combined_weights.sum()  # 총합 1로
combined_weights = combined_weights.sort_values(ascending=False)

# [7] 결과 출력
print("최종 MVP 포트폴리오 구성 (비중 기반, 총 10개 자산):")
print(combined_weights)

# `Black-litterman`
- 시장 균형 수익률(예: CAPM 기반)과 투자자의 주관적 뷰를 통합하여 기대 수익률을 산정함.
- 포트폴리오 비중을 추정할 때 시장 데이터 + 투자자의 의견을 결합해 더 안정적이고 현실적인 비중을 도출함.
- 블랙-리터먼(Black-Litterman) 모델은 전통적인 평균-분산 모델(MVO)의 한계를 보완함.

In [ ]:
# import numpy as np
# import pandas as pd
# from numpy.linalg import inv

# def black_litterman_constrained(
#     returns_3yr_short,
#     returns_3yr_mid,
#     returns_3yr_long,   
#     returns_bond, 
#     view_asset1='BTC', 
#     view_asset2='ETH', 
#     view_diff=0.02,
#     num_assets=10,
#     bond_weight=0.4,
#     other_weight=0.6,
#     tau=0.05,
#     epsilon=1e-5
# ):
#     """
#     Black-Litterman 모델: 수식 기반 + 제약 조건 포함 (자산 수/채권 가중치 지정 가능)
#     """

#     # 1. 수익률 통합
#     returns_all = pd.concat([returns_3yr_short,returns_3yr_long,returns_3yr_mid, returns_bond], axis=1).dropna()
#     pi = returns_all.mean().values
#     cov = returns_all.cov().values
#     cov_reg = cov + epsilon * np.eye(cov.shape[0])  # 정칙화
#     assets = returns_all.columns.tolist()
#     n = len(assets)

#     # 2. 투자자 뷰 설정 (Q = A - B = view_diff)
#     P = np.zeros((1, n))
#     P[0, assets.index(view_asset1)] = 1
#     P[0, assets.index(view_asset2)] = -1
#     Q = np.array([view_diff])
#     omega = np.diag([0.0025])  # 불확실성

#     # 3. Black-Litterman 수식 기반 posterior return
#     middle = inv(P @ (tau * cov_reg) @ P.T + omega)
#     bl_return = pi + tau * cov_reg @ P.T @ middle @ (Q - P @ pi)

#     # 4. 최적 비중 계산 (MVO)
#     inv_cov = inv(cov_reg)
#     raw_weights = inv_cov @ bl_return
#     raw_weights /= np.sum(raw_weights)

#     # 5. 결과 정리
#     df = pd.DataFrame({
#         'Asset': assets,
#         'Equilibrium Return': pi,
#         'Posterior Return (BL)': bl_return,
#         'Raw Weight': raw_weights
#     })

#     # 6. 상위 N개 자산만 선택
#     df_top = df.sort_values(by='Posterior Return (BL)', ascending=False).head(num_assets).copy()

#     # 7. 채권/기타 자산 분리
#     df_top['Group'] = df_top['Asset'].apply(lambda x: 'Bond' if x.startswith('BOND') else 'Other')
#     bond_count = (df_top['Group'] == 'Bond').sum()
#     other_count = num_assets - bond_count

#     # 8. 가중치 할당 (제약 기반)
#     def assign_weight(row):
#         if row['Group'] == 'Bond' and bond_count > 0:
#             return bond_weight / bond_count
#         elif row['Group'] == 'Other' and other_count > 0:
#             return other_weight / other_count
#         else:
#             return 0.0

#     df_top['BL Weight'] = df_top.apply(assign_weight, axis=1)

#     return df_top.reset_index(drop=True)

In [ ]:
# result = black_litterman_constrained(
#     returns_3yr_short, returns_bond,
#     view_asset1='BTC', view_asset2='ETH', view_diff=0.02,
#     num_assets=12,
#     bond_weight=0.3,
#     other_weight=0.7
# )
# print(result)

- BL Weight : 자산이 차지하는 투자 비중 

# `Risk Parity`

- 각 자산의 위험 기여도(Risk Contribution)를 동일하게 조정
- 수익률 예측 불필요 → 변동성 기반 포트폴리오
- 분산이 큰 자산 비중 ↓, 안정 자산 비중 ↑

**목표 함수**  
$$
\text{Minimize} \sum_i (RC_i - \bar{RC})^2
$$

**장점**  
- 수익률 예측 없어도 작동
- 분산 효과 우수 (보수적 운용 전략)

**단점**  
- 성장 자산 기회 상실 우려
- 채권 과도 비중 가능성 있음

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Risk Parity 최적화 함수
def risk_parity_portfolio(returns, max_assets=None):
    cov = returns.cov().values
    n_assets = cov.shape[0]
    columns = returns.columns

    def risk_contribution(weights):
        portfolio_var = weights.T @ cov @ weights
        marginal_contrib = cov @ weights
        risk_contrib = weights * marginal_contrib
        return risk_contrib / portfolio_var

    def risk_budget_objective(weights):
        rc = risk_contribution(weights)
        return np.sum((rc - rc.mean())**2)

    x0 = np.ones(n_assets) / n_assets
    cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bounds = tuple((0, 1) for _ in range(n_assets))

    result = minimize(risk_budget_objective, x0, method='SLSQP', bounds=bounds, constraints=cons)
    weights = pd.Series(result.x, index=columns)

    if max_assets:
        weights = weights.sort_values(ascending=False).head(max_assets)
        weights = weights / weights.sum()

    return weights.sort_values(ascending=False)

In [ ]:
# 설정: 총 10개 자산, 비중은 3yr:bond = 50:50
total_assets = 10
ratio_3yr = 0.5
ratio_bond = 0.5

# 비중 기반 자산 수 계산
max_assets_3yr = int(round(total_assets * ratio_3yr))
max_assets_bond = total_assets - max_assets_3yr

# Risk Parity 최적화 적용
rp_weights_3yr = risk_parity_portfolio(returns_3yr_short, max_assets=max_assets_3yr)
rp_weights_bond = risk_parity_portfolio(returns_bond, max_assets=max_assets_bond)

# 가중치 결합 및 재정규화
final_weights = pd.concat([rp_weights_3yr, rp_weights_bond])
final_weights = final_weights / final_weights.sum()  # 총합 1로 정규화
final_weights = final_weights.sort_values(ascending=False)

# 출력
print("최종 Risk Parity 포트폴리오 (총 10자산):")
print(final_weights)

# `Minimum Variance Portfolio`

- **목표**: 전체 자산군의 포트폴리오 분산(리스크)를 최소화하는 조합 탐색
- **수익률 예측 없이** 공분산 행렬을 기준으로 리스크 최소화를 추구함

#### 수학적 구조
- 목적 함수: min \( w^T \Sigma w \)
- 제약조건: \( \sum w_i = 1 \), \( w_i \geq 0 \)

#### 장점
- 시장 하락, 불확실한 환경에 유리
- 수익 예측 없이 운용 가능

#### 단점
- 수익 극대화 목적 아님
- 저수익 구조, 방어적 성향 강함

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

def minimum_variance_portfolio(returns, max_assets=None):
    """
    전체 자산 변동성을 최소화하는 MVP 모델 (최소 분산 포트폴리오)
    선택 자산 수를 max_assets로 제한할 수 있음
    """
    cov = returns.cov()
    columns = returns.columns

    # 초기 최적화 (전체 자산 대상)
    def portfolio_variance(w, cov_matrix):
        return w.T @ cov_matrix.values @ w

    n_assets = len(columns)
    x0 = np.ones(n_assets) / n_assets
    cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bounds = tuple((0, 1) for _ in range(n_assets))

    result = minimize(portfolio_variance, x0, args=(cov,), method='SLSQP', bounds=bounds, constraints=cons)
    weights = pd.Series(result.x, index=columns)

    # max_assets 제한: 상위 자산만 선택하여 다시 최적화
    if max_assets:
        selected = weights.sort_values(ascending=False).head(max_assets).index
        cov_sub = cov.loc[selected, selected]
        n_sel = len(selected)
        x0_sel = np.ones(n_sel) / n_sel
        bounds_sel = tuple((0, 1) for _ in range(n_sel))
        cons_sel = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

        result_sel = minimize(portfolio_variance, x0_sel, args=(cov_sub,), method='SLSQP', bounds=bounds_sel, constraints=cons_sel)
        weights_sel = pd.Series(result_sel.x, index=selected)
        return weights_sel.sort_values(ascending=False)

    return weights.sort_values(ascending=False)

In [ ]:
# 채권과 나머지 자산을 따로 MVP 적용
mvp_3yr = minimum_variance_portfolio(returns_3yr_mid, max_assets=100)
mvp_bond = minimum_variance_portfolio(returns_bond, max_assets=100)

# 비중 배분: 채권 70%, 나머지 30%
final_mvp_weights = pd.concat([
    mvp_3yr * 0.7,
    mvp_bond * 0.3
])

final_mvp_weights = final_mvp_weights[final_mvp_weights > 0.01].sort_values(ascending=False)
display(final_mvp_weights)

# `CVaR Optimization`

- **목표**: 신뢰수준 α에서 발생할 수 있는 **최악의 손실(CVaR)**을 최소화
- **핵심 전략**: 단순히 손실의 확률만 보지 않고, 최악의 상황에서 평균 손실까지 고려하는 포트폴리오 설계

#### 수학적 구조
- 목적 함수: 
$$
minimize CVaR = VaR + (1 / (1 - α)) * 평균 초과손실
$$

- 제약조건:
    - 자산 비중 합 = 1
    - 초과손실 z ≥ 0

#### 장점
- 금융위기, 블랙스완 등의 극단적 상황에서도 유리
- 수익률보다 리스크 회피가 우선인 전략에 적합

#### 단점
- 계산 복잡도 높음
- 수익률 극대화보다 보수적인 결과 도출 가능


### `α`
- 알파(α) = 0.95 → 95%의 신뢰수준. 
- 즉, 전체 투자 기간 중 95%의 경우에는 손실이 α 수준 이하로 발생할 것이라는 의미입니다.
- 반대로, 5%의 확률로는 α 수준보다 더 큰 손실이 발생할 수 있다는 뜻이기도 합니다.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# CVaR 최적화 함수
def cvar_optimization(returns, alpha=0.95, max_assets=None):
    T, N = returns.shape
    columns = returns.columns
    R = returns.values

    total_vars = N + T + 1  # w (N) + VaR (1) + z (T)

    # 목적 함수 계수 설정
    c = np.zeros(total_vars)
    c[N] = 1  # VaR
    c[N+1:] = 1 / ((1 - alpha) * T)  # z

    # 불평등 제약 (시나리오 제약)
    A_ub = np.zeros((T, total_vars))
    b_ub = np.zeros(T)
    for t in range(T):
        A_ub[t, :N] = -R[t]
        A_ub[t, N] = -1
        A_ub[t, N+1+t] = -1
        b_ub[t] = 0

    # w의 합 = 1 제약
    A_eq = np.zeros((1, total_vars))
    A_eq[0, :N] = 1
    b_eq = [1]

    # 경계조건: 0 ≤ w_i ≤ 1, z ≥ 0
    bounds = [(0, 1)] * N + [(-np.inf, np.inf)] + [(0, None)] * T

    res = linprog(c, A_ub, b_ub, A_eq, b_eq, bounds=bounds, method='highs')
    weights = pd.Series(res.x[:N], index=columns)

    if max_assets:
        weights = weights.sort_values(ascending=False).head(max_assets)
        weights = weights / weights.sum()

    return weights.sort_values(ascending=False)

In [ ]:
# 설정: 총 자산 수와 비중 비율
total_assets = 10
ratio_3yr = 0.4
ratio_bond = 0.6

max_assets_3yr = int(round(total_assets * ratio_3yr))
max_assets_bond = total_assets - max_assets_3yr

# CVaR 최적화 수행
cvar_weights_3yr = cvar_optimization(returns_3yr_short, alpha=0.95, max_assets=max_assets_3yr)
cvar_weights_bond = cvar_optimization(returns_bond, alpha=0.95, max_assets=max_assets_bond)

# 비중 합산 및 정규화
combined_weights = pd.concat([
    cvar_weights_3yr * ratio_3yr,
    cvar_weights_bond * ratio_bond
])
combined_weights = combined_weights / combined_weights.sum()  # 총합 1로
combined_weights = combined_weights.sort_values(ascending=False)

# 출력
print("최종 CVaR 최적화 포트폴리오 (총 10자산):")
print(combined_weights)

- 페르소나0- 코인10 채권30 나머지 60 (마르코프위츠)
- 페르소나1- 채권40 나머지(코인제외) 60 (마르코프위츠)
- 페르소나2- 채권50 나머지(코인제외) 50 (민 바란스)
- 페르소나3- 나머지(코인포함)100 채권,x (cvar)
- 페르소나4- 나머지 30(코인제외) + 완전방어70 (채권) (리스크패러티)

# 페르소나 0

In [ ]:
# [3] 설정: 총 자산 수와 비중 기반 분할
total_assets = 10
ratio_3yr = 0.7
ratio_bond = 0.3

max_assets_3yr = int(round(total_assets * ratio_3yr))
max_assets_bond = total_assets - max_assets_3yr

# [4] 수익률 표준화
returns_3yr_scaled = scale_returns(returns_3yr_mid)
returns_bond_scaled = scale_returns(returns_bond)

# [5] MVP 최적화
weights_3yr = minimum_variance_portfolio(returns_3yr_scaled, max_assets=max_assets_3yr)
weights_bond = minimum_variance_portfolio(returns_bond_scaled, max_assets=max_assets_bond)

# [6] 비중 합산 및 정규화
combined_weights = pd.concat([
    weights_3yr * ratio_3yr,
    weights_bond * ratio_bond
])
combined_weights = combined_weights / combined_weights.sum()  # 총합 1로
combined_weights = combined_weights.sort_values(ascending=False)

# [7] 결과 출력
print("최종 MVP 포트폴리오 구성 (비중 기반, 총 10개 자산):")
print(combined_weights)

In [ ]:
# [9-2] 바 차트 시각화
plt.figure(figsize=(16, 12))
combined_weights.plot(kind='bar')
plt.title('최종 포트폴리오 비중 (Bar Chart)')
plt.xlabel('자산명')
plt.ylabel('비중 (%)')
plt.xticks(rotation=45, ha='right', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 페르소나 1

In [ ]:
# [4] 표준화
returns_3yr_scaled = scale_returns(returns_3yr_mid)
returns_bond_scaled = scale_returns(returns_bond)

# [5] MVP 최적화 (각 자산군 내)
weights_3yr = minimum_variance_portfolio(returns_3yr_scaled, max_assets=5)
weights_bond = minimum_variance_portfolio(returns_bond_scaled, max_assets=5)

# [6] 비중 반영 (채권: 40%, 기타 자산: 60%)
weights_3yr_final = weights_3yr * 0.6
weights_bond_final = weights_bond * 0.4

# [7] 통합 포트폴리오 생성
final_portfolio1 = pd.concat([weights_3yr_final, weights_bond_final])
final_portfolio1 = final_portfolio1[final_portfolio1 > 0.01].sort_values(ascending=False)

# [8] 결과 출력
print("최종 포트폴리오 (자산별 가중치):")
print(final_portfolio1)

In [ ]:
# [3] 설정: 총 자산 수와 비중 기반 분할
total_assets = 10
ratio_3yr = 0.6
ratio_bond = 0.4

max_assets_3yr = int(round(total_assets * ratio_3yr))
max_assets_bond = total_assets - max_assets_3yr

# [4] 수익률 표준화
returns_3yr_scaled = scale_returns(returns_3yr_mid)
returns_bond_scaled = scale_returns(returns_bond)

# [5] MVP 최적화
weights_3yr = minimum_variance_portfolio(returns_3yr_scaled, max_assets=max_assets_3yr)
weights_bond = minimum_variance_portfolio(returns_bond_scaled, max_assets=max_assets_bond)

# [6] 비중 합산 및 정규화
combined_weights = pd.concat([
    weights_3yr * ratio_3yr,
    weights_bond * ratio_bond
])
combined_weights = combined_weights / combined_weights.sum()  # 총합 1로
combined_weights = combined_weights.sort_values(ascending=False)

# [7] 결과 출력
print("최종 MVP 포트폴리오 구성 (비중 기반, 총 10개 자산):")
print(combined_weights)

In [ ]:
# [9-2] 바 차트 시각화
plt.figure(figsize=(16, 12))
combined_weights.plot(kind='bar')
plt.title('최종 포트폴리오 비중 (Bar Chart)')
plt.xlabel('자산명')
plt.ylabel('비중 (%)')
plt.xticks(rotation=45, ha='right', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 페르소나 2

In [ ]:
# [3] 설정: 총 자산 수와 비중 기반 분할
total_assets = 10
ratio_3yr = 0.5
ratio_bond = 0.5

max_assets_3yr = int(round(total_assets * ratio_3yr))
max_assets_bond = total_assets - max_assets_3yr

# [4] 수익률 표준화
returns_3yr_scaled = scale_returns(returns_3yr_short)
returns_bond_scaled = scale_returns(returns_bond)

# [5] MVP 최적화
weights_3yr = minimum_variance_portfolio(returns_3yr_scaled, max_assets=max_assets_3yr)
weights_bond = minimum_variance_portfolio(returns_bond_scaled, max_assets=max_assets_bond)

# [6] 비중 합산 및 정규화
combined_weights = pd.concat([
    weights_3yr * ratio_3yr,
    weights_bond * ratio_bond
])
combined_weights = combined_weights / combined_weights.sum()  # 총합 1로
combined_weights = combined_weights.sort_values(ascending=False)

# [7] 결과 출력
print("최종 MVP 포트폴리오 구성 (비중 기반, 총 10개 자산):")
print(combined_weights)

In [ ]:
# [9-2] 바 차트 시각화
plt.figure(figsize=(16, 12))
combined_weights.plot(kind='bar')
plt.title('최종 포트폴리오 비중 (Bar Chart)')
plt.xlabel('자산명')
plt.ylabel('비중 (%)')
plt.xticks(rotation=45, ha='right', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 페르소나 3

In [ ]:
# 설정: 총 자산 수와 비중 비율
total_assets = 10
ratio_3yr = 0.9
ratio_bond = 0.1

max_assets_3yr = int(round(total_assets * ratio_3yr))
max_assets_bond = total_assets - max_assets_3yr

# CVaR 최적화 수행
cvar_weights_3yr = cvar_optimization(returns_3yr_long, alpha=0.90, max_assets=max_assets_3yr)
cvar_weights_bond = cvar_optimization(returns_bond, alpha=0.90, max_assets=max_assets_bond)

# 비중 합산 및 정규화
combined_weights = pd.concat([
    cvar_weights_3yr * ratio_3yr,
    cvar_weights_bond * ratio_bond
])
combined_weights = combined_weights / combined_weights.sum()  # 총합 1로
combined_weights = combined_weights.sort_values(ascending=False)

# 출력
print("최종 CVaR 최적화 포트폴리오 (총 10자산):")
print(combined_weights)

In [ ]:
# [9-2] 바 차트 시각화
plt.figure(figsize=(16, 12))
combined_weights.plot(kind='bar')
plt.title('최종 포트폴리오 비중 (Bar Chart)')
plt.xlabel('자산명')
plt.ylabel('비중 (%)')
plt.xticks(rotation=45, ha='right', fontsize = 14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 페르소나 4

In [ ]:
# 설정: 총 자산 수와 비중 비율
total_assets = 10
ratio_3yr = 0.3
ratio_bond = 0.7

max_assets_3yr = int(round(total_assets * ratio_3yr))
max_assets_bond = total_assets - max_assets_3yr

# CVaR 최적화 수행
cvar_weights_3yr = cvar_optimization(returns_3yr_short, alpha=0.95, max_assets=max_assets_3yr)
cvar_weights_bond = cvar_optimization(returns_bond, alpha=0.95, max_assets=max_assets_bond)

# 비중 합산 및 정규화
combined_weights = pd.concat([
    cvar_weights_3yr * ratio_3yr,
    cvar_weights_bond * ratio_bond
])
combined_weights = combined_weights / combined_weights.sum()  # 총합 1로
combined_weights = combined_weights.sort_values(ascending=False)

# 출력
print("최종 CVaR 최적화 포트폴리오 (총 10자산):")
print(combined_weights)

In [ ]:
# [9-2] 바 차트 시각화
plt.figure(figsize=(16, 12))
combined_weights.plot(kind='bar')
plt.title('최종 포트폴리오 비중 (Bar Chart)')
plt.xlabel('자산명')
plt.ylabel('비중 (%)')
plt.xticks(rotation=45, ha='right', fontsize = 14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()